In [15]:
con.close()

# Load all three layers into DuckDB

In [16]:
import duckdb

con = duckdb.connect(
    r"D:\pyspark_vs_code\medallion_project_duckdb\database\sales_db.duckdb"
)
#Here sales_db would be the catalog name.and bronze will scheam/database and under this table will create.

# Create schemas
con.sql("CREATE SCHEMA IF NOT EXISTS bronze")
con.sql("CREATE SCHEMA IF NOT EXISTS silver")
con.sql("CREATE SCHEMA IF NOT EXISTS gold")

CountBIGINT


In [17]:
con.sql("show schemas").show()
con.sql(" use bronze")
con.sql("show tables")

┌───────────────┬─────────────┬─────────┐
│ database_name │ schema_name │ current │
│    varchar    │   varchar   │ boolean │
├───────────────┼─────────────┼─────────┤
│ sales_db      │ bronze      │ false   │
│ sales_db      │ gold        │ false   │
│ sales_db      │ main        │ true    │
│ sales_db      │ silver      │ false   │
└───────────────┴─────────────┴─────────┘



nameVARCHAR
raw_sales


# Load the Parquet datasets into permanent DuckDB tables:

In [18]:
base_path = "D:/pyspark_vs_code/medallion_project_duckdb/data"

for schema_name, table in [
    ("bronze", "raw_sales"),
    ("silver", "clean_sales"),
    ("gold", "state_sales")
]:
    con.sql(f"""
        CREATE OR REPLACE TABLE {schema_name}.{table} AS
        SELECT *
        FROM read_parquet(
            '{base_path}/{schema_name}/*.parquet'
        )
    """)

# query tables using three-part names:

In [19]:
con.sql("""
    SELECT *
    FROM sales_db.gold.state_sales
    ORDER BY total_sales DESC
""").show()

┌──────────────────────┬───────────────┬──────────────┐
│        state         │  total_sales  │ total_orders │
│       varchar        │ decimal(23,2) │    int64     │
├──────────────────────┼───────────────┼──────────────┤
│ California           │    1334534.23 │         7268 │
│ Texas                │    1039762.89 │         6406 │
│ Florida              │     532213.01 │         4095 │
│ Oregon               │     366316.69 │         3005 │
│ Ohio                 │     359566.70 │         2981 │
│ Colorado             │     355114.52 │         2919 │
│ Tennessee            │     352222.28 │         2938 │
│ Washington           │     345998.70 │         2947 │
│ Virginia             │     268891.32 │         2256 │
│ Massachusetts        │     267025.67 │         2268 │
│  ·                   │         ·     │           ·  │
│  ·                   │         ·     │           ·  │
│  ·                   │         ·     │           ·  │
│ Utah                 │     168689.40 │        

In [20]:
con.sql("""
    SELECT *
    FROM gold.state_sales
""").show()

┌────────────────┬───────────────┬──────────────┐
│     state      │  total_sales  │ total_orders │
│    varchar     │ decimal(23,2) │    int64     │
├────────────────┼───────────────┼──────────────┤
│ Utah           │     168689.40 │         1538 │
│ Hawaii         │      85340.69 │          814 │
│ Minnesota      │     181497.34 │         1590 │
│ Ohio           │     359566.70 │         2981 │
│ Oregon         │     366316.69 │         3005 │
│ Texas          │    1039762.89 │         6406 │
│ Pennsylvania   │     178550.70 │         1604 │
│ Connecticut    │     177010.04 │         1599 │
│ Nebraska       │     177600.32 │         1601 │
│ Nevada         │     182117.16 │         1620 │
│   ·            │          ·    │           ·  │
│   ·            │          ·    │           ·  │
│   ·            │          ·    │           ·  │
│ Mississippi    │      90551.17 │          840 │
│ Tennessee      │     352222.28 │         2938 │
│ Florida        │     532213.01 │         4095 │


### Here I write the same query and filter the data for total order which is more than 1000.
- you can also keep building on a relation without writing more SQL strings:

In [21]:
rel = con.sql ("SELECT *FROM gold.state_sales ")
#here i will reuse the define object for filteration of order which is more than 1000
result_final = rel.filter("total_orders>1000").limit(3)
result_final.show() 

┌───────────┬───────────────┬──────────────┐
│   state   │  total_sales  │ total_orders │
│  varchar  │ decimal(23,2) │    int64     │
├───────────┼───────────────┼──────────────┤
│ Utah      │     168689.40 │         1538 │
│ Minnesota │     181497.34 │         1590 │
│ Ohio      │     359566.70 │         2981 │
└───────────┴───────────────┴──────────────┘



In [22]:
rel = con.sql("SELECT * FROM gold.state_sales")

result = (
    rel.filter("total_orders > 1000")
       .project("*, total_sales / total_orders AS avg_order_value")
       .order("avg_order_value DESC")
       .limit(3)
)
result.show()

┌────────────┬───────────────┬──────────────┬────────────────────┐
│   state    │  total_sales  │ total_orders │  avg_order_value   │
│  varchar   │ decimal(23,2) │    int64     │       double       │
├────────────┼───────────────┼──────────────┼────────────────────┤
│ California │    1334534.23 │         7268 │ 183.61780820033022 │
│ Texas      │    1039762.89 │         6406 │  162.3107852013737 │
│ Florida    │     532213.01 │         4095 │ 129.96654700854702 │
└────────────┴───────────────┴──────────────┴────────────────────┘



## Window functions
- Window functions are plain SQL in DuckDB. In the Relation API you put them inside project().

In [23]:
# Top state by sales in each region
rel = con.sql("SELECT * FROM gold.state_sales")
#rel.show()
ranked = rel.project("""
*,
row_number() over (order by total_sales) as rn,
sum(total_sales) over(order by total_sales) as sales_region,
lag(total_sales) over ( order by total_sales) as prev_sales
""")
ranked.filter("rn<=10").show()
final = ranked.project(
    """
    state,total_sales,rn,sales_region,prev_sales
    
    """
)
final.limit(5).show()

┌──────────────────────┬───────────────┬──────────────┬───────┬───────────────┬───────────────┐
│        state         │  total_sales  │ total_orders │  rn   │ sales_region  │  prev_sales   │
│       varchar        │ decimal(23,2) │    int64     │ int64 │ decimal(38,2) │ decimal(23,2) │
├──────────────────────┼───────────────┼──────────────┼───────┼───────────────┼───────────────┤
│ Hawaii               │      85340.69 │          814 │     1 │      85340.69 │          NULL │
│ Iowa                 │      86821.53 │          812 │     2 │     172162.22 │      85340.69 │
│ Maryland             │      86881.32 │          829 │     3 │     259043.54 │      86821.53 │
│ Idaho                │      87323.57 │          810 │     4 │     346367.11 │      86881.32 │
│ Indiana              │      88237.96 │          827 │     5 │     434605.07 │      87323.57 │
│ Oklahoma             │      90169.20 │          823 │     6 │     524774.27 │      88237.96 │
│ Mississippi          │      90551.17 │

In [24]:
con.sql("""
    SELECT
        table_schema,
        table_name
    FROM information_schema.tables
    WHERE table_schema IN ('bronze', 'silver', 'gold')
    ORDER BY table_schema, table_name
""").show()

┌──────────────┬─────────────┐
│ table_schema │ table_name  │
│   varchar    │   varchar   │
├──────────────┼─────────────┤
│ bronze       │ raw_sales   │
│ gold         │ state_sales │
│ silver       │ clean_sales │
└──────────────┴─────────────┘



# compare record counts:

In [25]:
con.sql("""
    SELECT 'bronze' AS layer, COUNT(*) AS records
    FROM bronze.raw_sales

    UNION ALL

    SELECT 'silver', COUNT(*)
    FROM silver.clean_sales

    UNION ALL

    SELECT 'gold', COUNT(*)
    FROM gold.state_sales
""").show()

┌─────────┬─────────┐
│  layer  │ records │
│ varchar │  int64  │
├─────────┼─────────┤
│ bronze  │   95904 │
│ silver  │   95904 │
│ gold    │      38 │
└─────────┴─────────┘



## fetch silver layer

In [26]:
con.sql("select * from sales_db.silver.clean_sales limit 4").show()

┌────────┬────────────┬──────────┬───────────────┬────────────────────────┬───────────────┬──────────────┬────────────┬──────────────┬────────────────────────────┬───────────────────────────────────────────────────────────────────┐
│ row_id │ order_date │ order_id │     sales     │        category        │ sub_category  │     city     │   state    │ payment_type │       ingestion_time       │                            source_file                            │
│ int64  │    date    │  int64   │ decimal(12,2) │        varchar         │    varchar    │   varchar    │  varchar   │   varchar    │         timestamp          │                              varchar                              │
├────────┼────────────┼──────────┼───────────────┼────────────────────────┼───────────────┼──────────────┼────────────┼──────────────┼────────────────────────────┼───────────────────────────────────────────────────────────────────┤
│    289 │ 2011-06-19 │  4000674 │        155.55 │ Outdoor Play Equipmen

In [27]:
col_fil = con.sql("select row_id,order_date,sales,category,sub_category,city,state,payment_type  from sales_db.silver.clean_sales ")


# 1. Window functions

In [28]:
rnkd = (col_fil
        .project("""

state,city,sales,
ROW_NUMBER() OVER(PARTITION BY state order by sales desc) as rnk,
lag(sales) over (partition by state order by sales) as prev_sale


""")
        
        )
rnkd.filter("rnk==1").show()

┌──────────────────────┬───────────────┬───────────────┬───────┬───────────────┐
│        state         │     city      │     sales     │  rnk  │   prev_sale   │
│       varchar        │    varchar    │ decimal(12,2) │ int64 │ decimal(12,2) │
├──────────────────────┼───────────────┼───────────────┼───────┼───────────────┤
│ Illinois             │ Chicago       │        199.91 │     1 │        199.87 │
│ Ohio                 │ Dayton        │        199.94 │     1 │        199.90 │
│ Oklahoma             │ Oklahoma City │        199.98 │     1 │        199.70 │
│ Indiana              │ Indianapolis  │        199.81 │     1 │        199.43 │
│ Oregon               │ Portland      │        199.84 │     1 │        199.83 │
│ Tennessee            │ Chattanooga   │        199.98 │     1 │        199.98 │
│ Michigan             │ Flint         │        199.84 │     1 │        199.79 │
│ Louisiana            │ New Orleans   │        199.62 │     1 │        199.37 │
│ Iowa                 │ Des

In [29]:
con.sql("""
SELECT * FROM sales_db.silver.clean_sales
QUALIFY ROW_NUMBER() OVER (PARTITION BY state ORDER BY sales DESC) 
= 1
""").show()

┌────────┬────────────┬──────────┬───────────────┬────────────────────┬────────────────────────┬────────────────┬────────────────┬──────────────┬────────────────────────────┬───────────────────────────────────────────────────────────────────┐
│ row_id │ order_date │ order_id │     sales     │      category      │      sub_category      │      city      │     state      │ payment_type │       ingestion_time       │                            source_file                            │
│ int64  │    date    │  int64   │ decimal(12,2) │      varchar       │        varchar         │    varchar     │    varchar     │   varchar    │         timestamp          │                              varchar                              │
├────────┼────────────┼──────────┼───────────────┼────────────────────┼────────────────────────┼────────────────┼────────────────┼──────────────┼────────────────────────────┼───────────────────────────────────────────────────────────────────┤
│  24867 │ 2011-11-01 │  400

In [32]:
rel_final = (con.sql("select * from sales_db.silver.clean_sales")
       .filter("state='Washington'")
       .project("state,city,sales")
       .order ("sales desc")
       )
rel_final.show()


ConnectionException: Connection Error: Connection has already been closed

In [33]:
con.close()

In [1]:
#create dataframe and fetch data this will access inmemory database.
import duckdb

con = duckdb.connect(":memory:")

df = con.execute("""
    SELECT *
    FROM (
        VALUES
            (101, 'Rahul', 50000),
            (102, 'Priya', 60000),
            (103, 'Amit', 70000)
    ) AS emp(emp_id, emp_name, salary)
""").fetchdf()

display(df)

,emp_id,emp_name,salary
0,101,Rahul,50000
1,102,Priya,60000
2,103,Amit,70000


In [3]:
con.execute("""
    SELECT *
    FROM (
        VALUES
            (101, 'Rahul', 50000),
            (102, 'Priya', 60000),
            (103, 'Amit', 70000)
    ) AS emp(emp_id, emp_name, salary)
""").show()

AttributeError: 'DuckDBPyConnection' object has no attribute 'show'